# Pielach Visualisation Masterthesis 2025

In [1]:
#imports
import ipywidgets as widgets
from ipywidgets.widgets import * 
from IPython.display import display, HTML, clear_output
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from rasterio.plot import show
from IPython.display import Javascript
import os

In [4]:
# --- Imports ---
import os
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# --- Load model-viewer globally ---
display(Javascript("""
if (!window.modelViewerLoaded) {
    const script = document.createElement('script');
    script.type = 'module';
    script.src = 'https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js';
    document.head.appendChild(script);
    window.modelViewerLoaded = true;
}
"""))

#  --- ASSETS---

# --- Kanalmigration models ---
waterdepth_models = {
    "2014": "static/2014.glb",
    "2015": "static/2015.glb",
    "2016": "static/2016.glb",
    "2021": "static/2021.glb",
    "2023": "static/2023.glb",
    "2024": "static/2024.glb"
}

# --- Erosion vs. Deposition models ---
change_models = {
    "2014-2015": "static/2015-2014.glb",
    "2015-2016": "static/2016-2015.glb",
    "2016-2021": "static/2021-2016.glb",
    "2021-2023": "static/2023-2021.glb",
    "2023-2024": "static/2024-2023_new3.glb",
    "2014-2016": "static/2016-2014_x.glb",
    "2014-2021": "static/2021-2014_x.glb",
    "2014-2023": "static/2023-2014_x.glb",
    "2014-2024": "static/2024-2014_x.glb",
    "2015-2021": "static/2021-2015_x.glb",
    "2015-2023": "static/2023-2015_x.glb",
    "2015-2024": "static/2024-2015_x.glb",
    "2016-2023": "static/2023-2016_x.glb",
    "2016-2024": "static/2024-2016_x.glb",
    "2021-2024": "static/2024-2021_x.glb"
}

# --- Legends for Erosion vs. Deposition models ---
change_legends = {
    "2014-2015": "static/legend_dods.png",
    "2015-2016": "static/legend_dods.png",
    "2016-2021": "static/legend_dods.png",
    "2021-2023": "static/legend_dods.png",
    "2023-2024": "static/legend_dod-2024-2023.png",
    "2014-2016": "static/legend_dods.png",
    "2014-2021": "static/legend_dods.png",
    "2014-2023": "static/legend_dods.png",
    "2014-2024": "static/legend_dod-2024-2023.png",
    "2015-2021": "static/legend_dods.png",
    "2015-2023": "static/legend_dods.png",
    "2015-2024": "static/legend_dod-2024-2023.png",
    "2016-2023": "static/legend_dods.png",
    "2016-2024": "static/legend_dod-2024-2023.png",
    "2021-2024": "static/legend_dod-2024-2023.png"
}

# --- Cross-section videos (10 total) ---
PERIODS = ["2014-2015", "2015-2016", "2016-2021", "2021-2023", "2023-2024"]
AREAS   = ["Ofenloch", "Neubacher Au"]
PERIOD_LABELS = {p: p.replace("-", " & ") for p in PERIODS}

cross_section_videos = {
    "Ofenloch": {
        "2014-2015": "static/ofenloch_2014-2015.mp4",
        "2015-2016": "static/ofenloch_2015-2016.mp4",
        "2016-2021": "static/ofenloch_2016-2021.mp4",
        "2021-2023": "static/ofenloch_2021-2023.mp4",
        "2023-2024": "static/ofenloch_2023-2024.mp4",
    },
    "Neubacher Au": {
        "2014-2015": "static/neubacherau_2014-2015.mp4",
        "2015-2016": "static/neubacherau_2015-2016.mp4",
        "2016-2021": "static/neubacherau_2016-2021.mp4",
        "2021-2023": "static/neubacherau_2021-2023.mp4",
        "2023-2024": "static/neubacherau_2023-2024.mp4",
    },
}

# --- UI ELEMENTS ---
# --- Title box ---
title_box = widgets.HTML("""
<div style="
    background-color: white;
    color: black;
    padding: 12px 12px;
    font-weight: bold;
    font-size: 20px;
    font-style: italic;
    display: inline-block;
    width: 90%;
    text-align: center;
">
PielachExplorer
</div>
""")

# --- Viewer toggle buttons ---
viewer_selector = widgets.ToggleButtons(
    options=["Übersicht", "Kanalmigration", "Erosion vs. Deposition", "Fluss-Querschnitte", "Info"],
    value="Übersicht",
    button_style='',
    tooltips=["Übersicht", "Kanalmigration", "Erosion vs. Deposition", "Fluss-Querschnitte", "Info"],
    layout=widgets.Layout(
        display='flex',
        flex_flow='row',
        justify_content='space-between',
        width='100%',
        height='40px'
    ),
    style={'button_width': 'auto'}
)

# --- Description box with legend ---
description_html = widgets.HTML()
legend_img = widgets.Image(format='png')
legend_img.layout = widgets.Layout(width='100px', height='auto', margin='0 auto', display='block')

description_box = widgets.VBox([description_html, legend_img], layout=widgets.Layout(
    padding='5px', width='100%', align_items='center'
))

def update_legend():
    if viewer_selector.value == "Erosion vs. Deposition":
        if change_slider_mode.value == "standard":
            selected_value = change_slider.value
            legend_img.layout.width = "100px"
        elif change_slider_mode.value == "Jahrevergleichen":
            start, end = change_range_slider.value
            selected_value = f"{start}-{end}"
            legend_img.layout.width = "100px"
        else:
            selected_value = "unknown"
            legend_img.layout.width = "100px"

        legend_file = change_legends.get(selected_value, "static/legend_dods.png")
        legend_img.layout.height = "auto"
        try:
            with open(legend_file, "rb") as f:
                legend_img.value = f.read()
        except:
            legend_img.value = b''

def update_description():
    if viewer_selector.value == "Kanalmigration":
        if water_slider.value in ["2023", "2024"]:
            description_html.value = (
                "In Folge von Starkregen wurde zwischen 15. und 16. September 2024 ein <b>100-jähriges Hochwasser</b> in der Pielach erreicht. "
                "Die Wasserdurchflusswerte betrugen bis zu <b>433m³/s</b> und führten unter anderem zu einer umfassenden Restrukturierung des Flussbettes. "
                "Die Auswirkungen auf die Erosion und Sedimentation sind besonders im Bereich <b>Neubacher Au</b> und <b>Ofenloch</b> deutlich sichtbar."
            )
        else:
            description_html.value = (
                "<p>Der Flusslauf zeigt eine charakteristische Abfolge aus:</p>"
                "<ul style='margin-left: 18px; padding-left: 0;'>"
                "<li><b>Riffles</b> – flache, schnell fließende Abschnitte</li>"
                "<li><b>Pools</b> – tiefere, ruhige Bereiche</li>"
                "</ul>"
                "Diese, sowie der gesamte laterale Flussverlauf verlagern sich kontinuierlich durch Erosion, Transport und Deposition von Sediment."
            )

        legend_img.layout.display = 'block'
        legend_img.layout.width = "100px"
        legend_img.layout.height = "auto"
        try:
            with open("static/legend_waterdepth.png", "rb") as f:
                legend_img.value = f.read()
        except:
            legend_img.value = b''
        description_box.children = [description_html, legend_img]  

    elif viewer_selector.value == "Erosion vs. Deposition":
        if change_slider_mode.value == "Jahrevergleichen":
            start, end = change_range_slider.value
            selected_value = f"{start}-{end}"
        else:
            selected_value = change_slider.value

        if selected_value == "2023-2024":
            description_html.value = (
                "In Folge von Starkregen wurde zwischen 15. und 16. September 2024 ein <b>100-jähriges Hochwasser</b> in der Pielach erreicht. "
                "Die Wasserdurchflusswerte betrugen bis zu <b>433m³/s</b> und führten zu erhöhten Erosions- und Depositionsraten. "
                "Die grossen Sedimentverschiebungen sind besonders im Bereich <b>Neubacher Au</b> und <b>Ofenloch</b> sehr gut beobachtbar."
            )
        else:
            legend_file = change_legends.get(selected_value, "static/legend_dods.png")
            try:
                with open(legend_file, "rb") as f:
                    import base64
                    encoded = base64.b64encode(f.read()).decode("utf-8")
                    img_html = f'<img src="data:image/png;base64,{encoded}" style="width:100px; height:auto; display:block; margin: 8px auto;" />'
            except:
                img_html = ''
        
            description_html.value = (
                "<div style='text-align: left; font-size: 13px;'>"
                "<ul style='margin-left: 10px; padding-left: 0px;'>"
                "<li><b>Erosion:</b> Wenn Wasser den Flussboden und Ufer abträgt und Sediment mit sich fortspült.</li>"
                "<li><b>Deposition:</b> Wenn Wasser Sediment mit sich bringt und als neue Schichten ablagert.</li>"
                "</ul>"
                f"{img_html}"
                "<p style='margin-top: 8px;'>Beides sind kontinuierliche Prozesse. Ihre Auswirkungen werden als Höhenunterschiede im Flussbett zwischen zwei Jahren sichtbar.</p>"
                "</div>"
            )
        
        legend_img.layout.display = 'none'


    elif viewer_selector.value == "Übersicht":
        description_html.value = (
            "<b>Die Pielach:</b> Ein voralpiner Fluss in Niederösterreich, der Teil des Natura 2000-Schutzgebietes ist."
            "<ul style='margin-left: 10px; padding-left: 0px;'>"
            "<li><b>Quelle:</b> ~1000 m.ü.M., Annaberg</li>"
            "<li><b>Länge:</b> ~68 km</li>"
            "<li><b>Mündung:</b> in die Donau bei Melk</li>"
            "<li><b>Morphologie:</b> mäandrierend-geflochtene bis mäandrierende Flussstruktur und <i>Riffle-Pool</i> Flusstyp</li>"
            "</ul>"
        )
        legend_img.layout.display = 'block'
        legend_img.layout.width = "180px"
        legend_img.layout.height = "auto"
        try:
            with open("static/pielach_nature.jpg", "rb") as f:
                legend_img.value = f.read()
        except:
            legend_img.value = b''
        description_box.children = [description_html, legend_img]  # <-- Fix here

    elif viewer_selector.value == "Fluss-Querschnitte":
        description_html.value = (
            "Die animierten Querschnitte zeigen Veränderungen des Flussprofils in der <b>Neubacher Au</b> und <b>beim Ofenloch</b> "
            "von 2014 bis 2024. Durch die <b>Gegenüberstellung von zwei Profilen</b>, lassen sich "
            "Deposition, Ufererosion und Profilverschiebungen anschaulich nachvollziehen.<br><br>"
            "Am stärksten zeigen sich die Veränderungen zwischen 2023 und 2024, als das <b>100-jährige Hochwasser</b> im September 2024 <b>vertikale Sedimentverschiebungen von bis zu 2.5m</b> verursachte."
        )
        legend_img.value = b''
        legend_img.layout.display = 'none'

    elif viewer_selector.value == "Info":
        description_html.value = (
            "<div style='font-size: 14px; line-height: 1.6;'>"
            "<p>Der <span style='font-style: italic;'>PielachExplorer</span> wurde von <b>Stefanie Germann</b> im Rahmen ihrer Masterarbeit im Erasmus Mundus Master of Cartography unter der Betreuung von Prof. G. Mandlburger (TU Wien) und Prof. A. Eltner (TU Dresden) entwickelt.</p>"
            "</div>"
        )
    
        legend_img.layout.display = 'block'
        legend_img.layout.width = "120px"
        legend_img.layout.height = "auto"
        try:
            with open("static/me.jpg", "rb") as f:
                legend_img.value = f.read()
        except:
            legend_img.value = b''
    
        description_box.children = [legend_img, description_html]  

    else:
        description_html.value = ""
        legend_img.value = b''
        legend_img.layout.display = 'none'
        

# --- Timesliders ---
water_slider = widgets.SelectionSlider(
    options=list(waterdepth_models.keys()),
    value="2014",
    description="Jahr:",
    continuous_update=False,
    layout=widgets.Layout(width="100%", margin="0 auto")
)

# --- Dropdown to switch between timeslider modes ---
change_slider_mode = widgets.Dropdown(
    options=[("Kontinuierlich ab 2014", "standard"), ("Zeitintervall wählen", "Jahrevergleichen")],
    value="standard",
    layout=widgets.Layout(width="auto"),
)

# --- Timeslider for "Kontinuierlich ab 2014" ---
change_slider = widgets.SelectionSlider(
    options=["2014-2015", "2015-2016", "2016-2021", "2021-2023", "2023-2024"],
    value="2014-2015",
    description="Zeitraum:",
    continuous_update=False,
    layout=widgets.Layout(width="100%", margin="0 auto")
)

# --- Timeslider for "Zeitinterwall wählen" ---
change_range_slider = widgets.SelectionRangeSlider(
    options=[2014, 2015, 2016, 2021, 2023, 2024],
    index=(0, 1),
    description="Zeitraum:",
    continuous_update=False,
    layout=widgets.Layout(width="100%", margin="0 auto")
)

# --- Model output and container ---
viewer_output = widgets.Output(layout=widgets.Layout(
    width='100%', overflow='visible', flex='0 0 auto', padding='0px', border='none'
))

slider_panel = widgets.VBox(layout=widgets.Layout(
    width='100%', height='0px', padding='0px', display='none'
))

# --- Viewers ---
# --- HTML model viewer function for Kanalmigration ---
def display_model_waterdepth(glb_path):
    return HTML(f"""
    <div class="mv-wrap" style="position: relative; width: 100%; height: 375px;">
      <model-viewer src="{glb_path}" alt="Waterdepth model"
        interaction-prompt="auto" interaction-prompt-threshold="8000"
        disable-default-lighting shadow-intensity="0" shadow-softness="0"
        exposure="0.003" environment-image="neutral" camera-controls 
        background-color="#FDF0F5" camera-orbit="0deg 0deg 7000m" 
        min-camera-orbit="auto auto 1m" max-camera-orbit="auto auto 10000m"
        field-of-view="45deg"
        style="width: 100%; height: 100%; filter: brightness(1.0) contrast(1.3);">
      </model-viewer>

      <div style="position: absolute; bottom: 10px; right: 10px; 
                  display: flex; flex-direction: column; align-items: flex-end; gap: 5px; z-index: 10;">
        <button data-zoomplus style="background-color: #f8f8f8; width: 25px; height: 25px;">+</button>
        <button data-zoomminus style="background-color: #f8f8f8; width: 25px; height: 25px;">-</button>
        <button data-reset style="background-color: #f8f8f8; width: 25px; height: 25px;">⟳</button>
        <button data-focus="1" style="background-color: #f8f8f8; width: 100px; height: 25px;">Neubacher Au</button>
        <button data-focus="2" style="background-color: #f8f8f8; width: 100px; height: 25px;">Ofenloch</button>
      </div>
    </div>

    <script>
    (function() {{
      const root = document.currentScript.previousElementSibling; // the .mv-wrap div
      const viewer = root.querySelector('model-viewer');

      root.querySelector('[data-zoomplus]').addEventListener('click', () => {{
        const o = viewer.getCameraOrbit(); viewer.cameraOrbit = `${{o.theta}}rad ${{o.phi}}rad ${{o.radius*0.9}}m`;
      }});
      root.querySelector('[data-zoomminus]').addEventListener('click', () => {{
        const o = viewer.getCameraOrbit(); viewer.cameraOrbit = `${{o.theta}}rad ${{o.phi}}rad ${{o.radius*1.1}}m`;
      }});
      root.querySelector('[data-reset]').addEventListener('click', () => {{
        viewer.cameraOrbit = "0deg 0deg 7000m";
        viewer.cameraTarget = "auto auto auto";
        viewer.fieldOfView = "45deg";
        viewer.jumpCameraToGoal();
      }});
      root.querySelectorAll('[data-focus]').forEach(btn => {{
        btn.addEventListener('click', () => {{
          const area = btn.getAttribute('data-focus');
          if (area === "1") {{
            viewer.cameraOrbit = "0deg 0deg 2300m";
            viewer.cameraTarget = "-850m 0m -450m";
          }} else {{
            viewer.cameraOrbit = "55deg 0deg 1800m";
            viewer.cameraTarget = "1000m 230m -250m";
          }}
          viewer.jumpCameraToGoal();
        }});
      }});
    }})();
    </script>
    """)

# --- HTML model viewer function for Erosion vs. Deposition ---
def display_model_change(glb_path):
    return HTML(f"""
    <div class="mv-wrap" style="position: relative; width: 100%; height: 375px;">
      <model-viewer src="{glb_path}" alt="Change model"
        interaction-prompt="auto" interaction-prompt-threshold="8000"
        disable-default-lighting shadow-intensity="0" shadow-softness="0"
        exposure="0.003" environment-image="neutral" camera-controls
        background-color="#FDF0F5" camera-orbit="0deg 0deg 7000m"
        min-camera-orbit="auto auto 1m" max-camera-orbit="auto auto 10000m"
        field-of-view="45deg"
        style="width: 100%; height: 100%; filter: brightness(0.9) contrast(1.3);">
      </model-viewer>

      <div style="position: absolute; bottom: 10px; right: 10px;
                  display: flex; flex-direction: column; align-items: flex-end; gap: 5px; z-index: 10;">
        <button data-zoomplus  style="background-color: #f8f8f8; width: 25px; height: 25px;">+</button>
        <button data-zoomminus style="background-color: #f8f8f8; width: 25px; height: 25px;">-</button>
        <button data-reset     style="background-color: #f8f8f8; width: 25px; height: 25px;">⟳</button>
        <button data-focus="1" style="background-color: #f8f8f8; width: 100px; height: 25px;">Neubacher Au</button>
        <button data-focus="2" style="background-color: #f8f8f8; width: 100px; height: 25px;">Ofenloch</button>
      </div>
    </div>

    <script>
    (function() {{
      const root = document.currentScript.previousElementSibling; // the .mv-wrap div
      const viewer = root.querySelector('model-viewer');

      const applyOrbit = (theta, phi, radius) => {{
        viewer.cameraOrbit = `${{theta}} ${{phi}} ${{radius}}`;
      }};

      root.querySelector('[data-zoomplus]').addEventListener('click', () => {{
        const o = viewer.getCameraOrbit();
        applyOrbit(o.theta + "rad", o.phi + "rad", (o.radius * 0.9) + "m");
      }});

      root.querySelector('[data-zoomminus]').addEventListener('click', () => {{
        const o = viewer.getCameraOrbit();
        applyOrbit(o.theta + "rad", o.phi + "rad", (o.radius * 1.1) + "m");
      }});

      root.querySelector('[data-reset]').addEventListener('click', () => {{
        viewer.cameraOrbit = "0deg 0deg 7000m";
        viewer.cameraTarget = "auto auto auto";
        viewer.fieldOfView = "45deg";
        viewer.jumpCameraToGoal();
      }});

      root.querySelectorAll('[data-focus]').forEach(btn => {{
        btn.addEventListener('click', () => {{
          const area = btn.getAttribute('data-focus');
          if (area === "1") {{
            viewer.cameraOrbit  = "0deg 30deg 2254m";
            viewer.cameraTarget = "-700m 28m 250m";
          }} else {{
            viewer.cameraOrbit  = "25deg 40deg 1800m";
            viewer.cameraTarget = "1200m -180m 220m";
          }}
          viewer.jumpCameraToGoal();
        }});
      }});
    }})();
    </script>
    """)

# --- Fluss-Querschnitte Viewer ---

def make_video_widget(video_path, prefer_url=True, max_height_px=330):
    v = widgets.Video(
        controls=True, autoplay=False, loop=False,
        layout=widgets.Layout(width="99%", height='370px', border="none")
    )
    if prefer_url and hasattr(v, "url"):
        v.url = video_path
        v.format = "mp4"
    else:
        if not os.path.exists(video_path):
            raise FileNotFoundError(f"Video not found: {video_path}")
        with open(video_path, "rb") as f:
            v.value = f.read()
        v.format = "mp4"
    return v

def build_cross_section_picker(initial_area="Neubacher Au", initial_period="2014-2015"):
    # Video
    video = make_video_widget(
        cross_section_videos[initial_area][initial_period],
        prefer_url=True,
        max_height_px=330
    )

    # Left: vertical radio buttons 
    area_radio = widgets.RadioButtons(
        options=["Neubacher Au", "Ofenloch"],
        value=initial_area if initial_area in ["Neubacher Au", "Ofenloch"] else "Neubacher Au",
        layout=widgets.Layout(width="140px", padding="0")   
    )

    # Right: period time slider (stretched)
    period_slider = widgets.SelectionSlider(
        options=[(PERIOD_LABELS[p], p) for p in PERIODS],  
        value=initial_period,
        description="Vergleich:",
        continuous_update=False,
        readout=True,
        layout=widgets.Layout(width="100%", min_width="0")
    )

    period_slider.style = {"description_width": "70px"}     

    # Layout row: [radio column] [slider] 
    controls_row = widgets.HBox(
        [
            widgets.VBox([area_radio], layout=widgets.Layout(width="140px", padding="0")),
            widgets.Box([period_slider], layout=widgets.Layout(flex="1 1 0%", padding="0 0 0 6px"))
        ],
        layout=widgets.Layout(width="100%", align_items="center")
    )

    # Update video helper
    def set_video(area, period):
        path = cross_section_videos[area][period]
        if hasattr(video, "url"):
            video.url = path
            video.format = "mp4"
        else:
            if not os.path.exists(path):
                display(HTML(f"<div style='color:#b00'>Datei nicht gefunden: {path}</div>"))
                return
            with open(path, "rb") as f:
                video.value = f.read()
            video.format = "mp4"

    # Wire events
    def on_area_change(change):
        if change["name"] == "value":
            set_video(change["new"], period_slider.value)

    def on_period_change(change):
        if change["name"] == "value":
            set_video(area_radio.value, change["new"])

    area_radio.observe(on_area_change, names="value")
    period_slider.observe(on_period_change, names="value")

    return widgets.VBox([video, controls_row], layout=widgets.Layout(width="100%"))

# --- Controller and layout ---

# --- Update viewer display ---
def update_viewer(change=None):
    with viewer_output:
        clear_output()
        selected = viewer_selector.value

        if selected == "Kanalmigration":
            viewer_box.layout.height = '375px'
            glb = waterdepth_models.get(water_slider.value)
            if glb:
                display(display_model_waterdepth(glb))

        elif selected == "Erosion vs. Deposition":
            viewer_box.layout.height = '375px'
            if change_slider_mode.value == "standard":
                selected_value = change_slider.value
            else:
                start, end = change_range_slider.value
                selected_value = f"{start}-{end}"
            glb = change_models.get(selected_value)
            if glb:
                display(display_model_change(glb))

        elif selected == "Übersicht":
            viewer_box.layout.height = '375px'
            display(HTML("""
            <div style="
                display: flex; 
                justify-content: center; 
                align-items: center; 
                height: 375px; 
                width: 100%;
                overflow: hidden;">
                <img src="static/overview_map.png" 
                     style="max-width: 100%; max-height: 100%; object-fit: contain;" />
            </div>
            """))
        
        elif selected == "Fluss-Querschnitte":
            viewer_box.layout.height = 'auto'
            wrapper = widgets.Box(
                [build_cross_section_picker(initial_area="Neubacher Au", initial_period="2014-2015")],
                layout=widgets.Layout(width='100%', overflow_x='hidden')
            )
            display(wrapper)


        elif selected == "Info":
            viewer_box.layout.height = '375px'
            display(info_popup)

# --- Sliders + timeline labels ---
def update_controls(change=None):
    selected = viewer_selector.value

    if selected == "Kanalmigration":
        slider_panel.children = [water_slider]
        slider_panel.layout.height = '30px'
        slider_panel.layout.display = 'flex'

    elif selected == "Erosion vs. Deposition":
        change_slider_mode.layout.width = '160px'
        if change_slider_mode.value == "standard":
            slider_row = widgets.HBox([
                change_slider_mode,
                widgets.Box([change_slider], layout=widgets.Layout(flex='1', padding='0 0 0 10px'))
            ], layout=widgets.Layout(width='100%', align_items='center'))
        else:
            slider_row = widgets.HBox([
                change_slider_mode,
                widgets.Box([change_range_slider], layout=widgets.Layout(flex='1', padding='0 0 0 10px'))
            ], layout=widgets.Layout(width='100%', align_items='center'))

        slider_panel.children = [slider_row]
        slider_panel.layout.height = '30px'
        slider_panel.layout.display = 'flex'

        update_description()
        update_viewer()
        return

    else:

        # Übersicht, Fluss-Querschnitte, Info 
        slider_panel.children = []
        slider_panel.layout.height = '0px'
        slider_panel.layout.display = 'none'

    update_description()
    update_viewer()

# --- Observe ---
viewer_selector.observe(update_controls, names='value')
water_slider.observe(update_viewer, names='value')
change_slider.observe(update_viewer, names='value')
change_slider.observe(lambda change: update_description(), names='value')
water_slider.observe(lambda change: update_description(), names='value')
change_range_slider.observe(update_viewer, names='value')
change_range_slider.observe(lambda change: update_description(), names='value')
change_slider_mode.observe(update_controls, names='value')
change_slider.observe(lambda change: update_legend(), names='value')
change_range_slider.observe(lambda change: update_legend(), names='value')
change_slider_mode.observe(lambda change: update_legend(), names='value')
viewer_selector.observe(lambda change: update_legend(), names='value')
change_range_slider.observe(update_viewer, names='value')
change_range_slider.observe(lambda change: update_description(), names='value')
change_range_slider.observe(lambda change: update_legend(), names='value')

# --- Left panel ---
left_panel = widgets.VBox([
    title_box,
    description_box
], layout=widgets.Layout(
    width='30%',
    padding='0px 5px 5px 5px',
    border='solid 1px lightgray',
    display='flex',
    flex_flow='column',
    height='450px'
))

left_panel.add_class('pielach-left')

global_style = widgets.HTML("""
<style>
  /* Tighten line-height for all text inside the left panel */
  .pielach-left, .pielach-left * { line-height: 1.6 !important; }
</style>
""")

# --- Right panel ---
viewer_box = widgets.Box(
    [viewer_output],
    layout=widgets.Layout(
        justify_content='center',
        align_items='center',
        display='flex',
        height='375px',              
        overflow='hidden',
        width='100%',
        padding='0px 0px 0px 20px'
    )
)

right_panel = widgets.VBox([
    viewer_selector,
    viewer_box,
    slider_panel
], layout=widgets.Layout(
    width='70%',
    height='450px',
    align_items='center',
    padding='0px'
))

# --- Main layout ---
inner_layout = widgets.HBox([left_panel, right_panel], layout=widgets.Layout(width='100%'))

outer_container = widgets.VBox([global_style, inner_layout], layout=widgets.Layout(
    border='20px solid #e0e0e0',
    padding='10px',
    margin='10px auto',
    border_radius='8px',
    width='850px',
    max_width='850px'
))


# --- Popups ---
welcome_text = widgets.HTML("""
<div style="font-size: 14px; line-height: 1.6; text-align: center;">
    <h2>Willkommen zum <span style="font-style: italic;">PielachExplorer</span>,</h2>
    <h4>Eine interaktive 3D-Visualisierung der Flussveränderungen 2014 - 2024.</h4>
    <p>Dieses Projekt basiert auf Messdaten der TU Wien und zeigt, wie sich die Pielach und ihr 
    Flussbett durch Sedimentbewegungen und Hochwasser verändert hat. Diese Veränderungen
    zu verstehen ist entscheidend für Hochwasserschutz, Naturschutz und Gewässermanagement.</p>
</div>
""")

welcome_close_btn = widgets.Button(
    description="Close",
    button_style='primary',
    icon='times',
    layout=widgets.Layout(width='100px')
)

welcome_popup = widgets.VBox(
    [welcome_text, welcome_close_btn],
    layout=widgets.Layout(
        border='1px solid',
        padding='15px',
        background_color='white',
        width='550px',
        margin='20px auto',
        align_items='center',
        box_shadow='0 4px 8px rgba(0,0,0,0.2)'
    )
)

info_popup = widgets.HTML("""
<div style="
    font-size: 14px;
    line-height: 1.6;
    padding-right: 20px;
    height: 350px;
    overflow-y: auto;
">
  <h3>Warum Fluss-Geomorphologie?</h3>
  <p>Flüsse gehören zu den dynamischsten Systemen der Erde. Fluss-Geomorphologie untersucht, wie Flüsse ihre Landschaft durch Erosion, Transport und Ablagerung von Sediment formen. Diese Prozesse prägen nicht nur Flussbette, sondern beeinflussen auch Hochwasserrisiken, landwirtschaftliche Flächen und die Lebensräume zahlreicher Tier- und Pflanzenarten.</p>

  <h3>Zweck & Bedeutung dieses Projekts</h3>
  <p>Der <i>PielachExplorer</i> zeigt die Veränderungen des Flussbettes zwischen 2014 und 2024. Mit 3D-Modellen, Animationen und verschiedenen Perspektiven lassen sich geomorphologische Prozesse (Erosion, Deposition, Kanalmigration) sowie die Folgen des 100-jährigen Hochwassers 2024 nachvollziehen. So wird deutlich, wie dieser ökologisch wertvolle Fluss sein Bett ständig umgestaltet. Ein besseres Verständnis der Flussdynamik ist entscheidend für Hochwasserschutz, ökologischen Gewässerausbau und Naturschutz und verdeutlicht zugleich, wie ein natürlicher Flussverlauf vielfältige Lebensräume prägt.</p>

  <h3>Datenerhebung</h3>
  <p>Die Basis des <i>PielachExplorer</i> bilden Messdaten, die mit einem speziellen Laserscanning-Verfahren (LiDAR) vom Departement für Geodäsie und Geoinformation der TU Wien erhoben wurden. Vom Flugzeug aus werden dabei Lichtimpulse ausgesendet und deren Rückkehr gemessen. Auf diese Weise entsteht ein hochauflösendes 3D-Modell von Flussbett und Landschaft, das Veränderungen des Flusses und seiner Lebensräume über mehrere Jahre sichtbar macht.</p>
  <p>Weitere Daten verwendet für die Umsetzung der Visualisierungen stammen von ©OpenStreetMap, ©MapTiler und dem Bundesamt für Eich- und Vermessungswesen (BEV), Österreich.
  
  <h3>Weiterführende Informationen</h3>
  <ul style="margin-left: 18px;">
    <li><a href="https://www.nf-int.org/sites/default/files/inline-files/NTs/Natura_Trail_Pielach.pdf" target="_blank" rel="noopener noreferrer">Pielach & Natura 2000 Infoblatt</a></li>
    <li><a href="https://link.springer.com/chapter/10.1007/978-3-662-59402-5_12" rel="noopener noreferrer">Mehr zu Fluss-Geomorphologie</a></li>
    <li><a href="https://www.mdpi.com/2072-4292/7/5/6160">Topobathymetrisches LiDAR – Fallstudie Pielach (in english)</a></li>
    <li><a href="https://www.weltkulturerbe-wachau.at/naturschutz/life-projekte/life-mostviertel-wachau" target="_blank" rel="noopener noreferrer">Pielach Renaturierung</a></li>
    <li><a href="https://www.bmluk.gv.at/themen/wasser/wasser-oesterreich/hydrographie/chronik-besonderer-ereignisse/hochwasser-september-2024.html" target="_blank" rel="noopener noreferrer">Artikel zum Hochwasserereignis September 2024</a></li>
    <li><a href="https://www.noe.gv.at/wasserstand/#/de/Messstellen" target="_blank" rel="noopener noreferrer">Wasserstandsnachrichten Niederösterreich</a></li>
    <li><a href="https://hora.gv.at/#/chwrz:-/bgrau/a-/@48.20313,15.41501,12z" target="_blank" rel="noopener noreferrer">HORA - Hochwasser Risiko Assessment</a></li>
  </ul>

  <h3>Kontakt</h3>
  <p>
    Fragen oder Feedback? → <a href="mailto:stefaniegermann@bluewin.ch">stefaniegermann@bluewin.ch</a>
  </p>
  </div>
</div>
""")

app_container = widgets.VBox(
    [welcome_popup],
    layout=widgets.Layout(
        align_items='center',
        width='850px'
    )
)

def close_welcome(_):
    app_container.children = [outer_container]

welcome_close_btn.on_click(close_welcome)

# --- Launch ---
update_controls()
display(app_container)


<IPython.core.display.Javascript object>